In [1]:
# ============================================================
# FedPIGD-B v2 (CLEAN FINAL VERSION)
# - No preprocessing visualization
# - GA-based client preprocessing
# - Signature-conditioned anchoring
# - Federated learning
# - Final .keras export
# ============================================================

import os
import time
import copy
import random
import json
import numpy as np
import tensorflow as tf

from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef
)
from sklearn.preprocessing import label_binarize

# ============================================================
# CONFIG
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TRAIN_DIR = "/Users/tanvir/Downloads/Research/Fish Recognition/Photos/Fish/train_data"
TEST_DIR  = "/Users/tanvir/Downloads/Research/Fish Recognition/Photos/Fish/test_data"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
LR = 1e-4
DROPOUT = 0.3

NUM_CLIENTS = 4
FL_ROUNDS = 10
LOCAL_EPOCHS_PER_ROUND = 1

EV_POP = 10
EV_GENS = 6
EV_SUBSET = 256
MAX_PIPE_LEN = 5

ANCHOR_LAMBDA = 1e-3

# ============================================================
# LOGGING
# ============================================================
def ts():
    return time.strftime("%H:%M:%S")

def log(msg):
    print(f"[{ts()}] [INFO] {msg}")

def phase(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

# ============================================================
# DATA LOADING
# ============================================================
phase("PHASE 0: LOAD DATA")

def list_classes(train_dir):
    classes = sorted([
        d for d in os.listdir(train_dir)
        if os.path.isdir(os.path.join(train_dir, d))
    ])
    return classes, {c: i for i, c in enumerate(classes)}

def load_paths_labels(data_dir, class_to_idx):
    paths, labels = [], []
    for cls, idx in class_to_idx.items():
        cls_path = os.path.join(data_dir, cls)
        for fn in os.listdir(cls_path):
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(cls_path, fn))
                labels.append(idx)
    return np.array(paths), np.array(labels)

def split_clients(paths, labels, n):
    idx = np.random.permutation(len(paths))
    paths, labels = paths[idx], labels[idx]

    proportions = np.random.dirichlet([1.0] * n)
    sizes = (proportions * len(paths)).astype(int)
    sizes[-1] = len(paths) - np.sum(sizes[:-1])

    clients = []
    s = 0
    for i, sz in enumerate(sizes):
        clients.append((paths[s:s+sz], labels[s:s+sz]))
        log(f"Client C{i+1}: {sz} samples")
        s += sz
    return clients

class_names, class_to_idx = list_classes(TRAIN_DIR)
K = len(class_names)

train_paths, train_labels = load_paths_labels(TRAIN_DIR, class_to_idx)
test_paths, test_labels = load_paths_labels(TEST_DIR, class_to_idx)

clients = split_clients(train_paths, train_labels, NUM_CLIENTS)

# ============================================================
# SAFE DECODER
# ============================================================
def decode_and_resize(path):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE)
    return tf.cast(img, tf.float32) / 255.0

# ============================================================
# NON-DIFFERENTIABLE PIPELINE (GA)
# ============================================================
OP_SPACE = [
    "IDENTITY", "GRAYSCALE", "AUTO_CONTRAST", "EQUALIZE",
    "SHARPEN", "BRIGHTNESS", "CONTRAST", "BLUR"
]

def sample_op():
    op = random.choice(OP_SPACE)
    p = {}
    if op in ["SHARPEN", "BRIGHTNESS", "CONTRAST"]:
        p["factor"] = float(np.random.uniform(0.8, 1.5))
    if op == "BLUR":
        p["radius"] = float(np.random.uniform(0.2, 1.5))
    return (op, p)

def apply_op(img, op):
    name, p = op
    if name == "GRAYSCALE":
        return ImageOps.grayscale(img).convert("RGB")
    if name == "AUTO_CONTRAST":
        return ImageOps.autocontrast(img)
    if name == "EQUALIZE":
        return ImageOps.equalize(img)
    if name == "SHARPEN":
        return ImageEnhance.Sharpness(img).enhance(p["factor"])
    if name == "BRIGHTNESS":
        return ImageEnhance.Brightness(img).enhance(p["factor"])
    if name == "CONTRAST":
        return ImageEnhance.Contrast(img).enhance(p["factor"])
    if name == "BLUR":
        return img.filter(ImageFilter.GaussianBlur(p["radius"]))
    return img

def apply_pipeline_np(img, pipe):
    img = Image.fromarray((img * 255).astype(np.uint8))
    for op in pipe:
        img = apply_op(img, op)
    return np.array(img).astype(np.float32) / 255.0

def tf_apply_pipeline(img, pipe):
    out = tf.numpy_function(
        lambda z: apply_pipeline_np(z, pipe),
        [img],
        tf.float32
    )
    out.set_shape([224, 224, 3])
    return out

# ============================================================
# MODEL
# ============================================================
def build_model(num_classes):
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )
    base.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dense(128, activation="relu", name="adapter")(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    out = tf.keras.layers.Dense(num_classes, activation="softmax", name="head")(x)

    return tf.keras.Model(base.input, out), base

global_model, backbone = build_model(K)
feat_model = tf.keras.Model(
    backbone.input,
    tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
)

# ============================================================
# GA FITNESS
# ============================================================
def separability_proxy(feats, labels):
    overall = feats.mean(axis=0)
    sb, sw = 0.0, 0.0
    for c in np.unique(labels):
        Xc = feats[labels == c]
        if len(Xc) < 2:
            continue
        mc = Xc.mean(axis=0)
        sb += len(Xc) * np.sum((mc - overall) ** 2)
        sw += np.sum((Xc - mc) ** 2)
    return sb / (sw + 1e-8)

def evolve_pipeline(paths, labels):
    pop = [[sample_op() for _ in range(random.randint(1, MAX_PIPE_LEN))]
           for _ in range(EV_POP)]
    best = None
    best_score = -1e9

    for _ in range(EV_GENS):
        scores = []
        idx = np.random.choice(len(paths), min(EV_SUBSET, len(paths)), False)
        for pipe in pop:
            xs = []
            for p in paths[idx]:
                img = decode_and_resize(tf.constant(p))
                img = tf_apply_pipeline(img, pipe)
                xs.append(img.numpy())
            feats = feat_model.predict(np.array(xs), verbose=0)
            score = separability_proxy(feats, labels[idx]) - 0.02 * len(pipe)
            scores.append((score, pipe))
            if score > best_score:
                best_score, best = score, pipe

        elites = [p for _, p in sorted(scores, reverse=True)[:EV_POP // 3]]
        pop = elites[:]
        while len(pop) < EV_POP:
            p = copy.deepcopy(random.choice(elites))
            p[random.randrange(len(p))] = sample_op()
            pop.append(p)
    return best

client_pipelines = [evolve_pipeline(cp, cl) for cp, cl in clients]

# ============================================================
# FEDERATED TRAINING
# ============================================================
def make_dataset(paths, labels, pipe):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def map_fn(x, y):
        img = decode_and_resize(x)
        img = tf_apply_pipeline(img, pipe)
        return img, y
    return ds.map(map_fn).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for r in range(FL_ROUNDS):
    log(f"FL ROUND {r+1}/{FL_ROUNDS}")
    global_weights = global_model.get_weights()
    updates, sizes = [], []

    for (cp, cl), pipe in zip(clients, client_pipelines):
        local, _ = build_model(K)
        local.set_weights(global_weights)
        local.compile(
            optimizer=tf.keras.optimizers.Adam(LR),
            loss="sparse_categorical_crossentropy"
        )
        local.fit(
            make_dataset(cp, cl, pipe),
            epochs=1,
            verbose=0
        )
        updates.append(local.get_weights())
        sizes.append(len(cp))

    new_weights = []
    for w in range(len(global_weights)):
        new_weights.append(
            sum(sizes[i] * updates[i][w] for i in range(NUM_CLIENTS)) / sum(sizes)
        )
    global_model.set_weights(new_weights)

# ============================================================
# EVALUATION (WITH TEST-TIME PREPROCESSING ADDED)
# ============================================================
phase("FINAL EVALUATION")

def make_test_dataset(paths, labels, pipe):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def map_fn(x, y):
        img = decode_and_resize(x)
        img = tf_apply_pipeline(img, pipe)   # <-- ADDED: apply preprocessing on test data
        return img, tf.cast(y, tf.int32)
    return ds.map(map_fn).batch(32).prefetch(tf.data.AUTOTUNE)

# Because you have multiple client pipelines, evaluate by averaging probabilities
# across all client pipelines (same model weights; only test-time preprocessing differs).
all_probs = []
for i, pipe in enumerate(client_pipelines):
    log(f"Testing with client pipeline C{i+1}")
    test_ds = make_test_dataset(test_paths, test_labels, pipe)
    probs = global_model.predict(test_ds, verbose=0)
    all_probs.append(probs)

pred_probs = np.mean(np.stack(all_probs, axis=0), axis=0)  # average over pipelines
preds = np.argmax(pred_probs, axis=1)

prec = precision_score(test_labels, preds, average="weighted", zero_division=0)
rec  = recall_score(test_labels, preds, average="weighted", zero_division=0)
ll   = log_loss(test_labels, pred_probs)

# one-hot for AUC metrics
y_true_bin = label_binarize(test_labels, classes=np.arange(K))

# ROC-AUC / PR-AUC handling (multiclass vs binary)
if K == 2:
    roc_auc = roc_auc_score(test_labels, pred_probs[:, 1])
    pr_auc  = average_precision_score(test_labels, pred_probs[:, 1])
else:
    roc_auc = roc_auc_score(
        y_true_bin, pred_probs,
        average="macro",
        multi_class="ovr"
    )
    pr_auc = average_precision_score(
        y_true_bin, pred_probs,
        average="macro"
    )

log(f"Accuracy     = {accuracy_score(test_labels, preds):.4f}")
log(f"Precision    = {prec:.4f} (weighted)")
log(f"Recall       = {rec:.4f} (weighted)")
log(f"F1-score     = {f1_score(test_labels, preds, average='weighted'):.4f}")
log(f"ROC-AUC      = {roc_auc:.4f}")
log(f"PR-AUC       = {pr_auc:.4f}")
log(f"Log-Loss     = {ll:.4f}")
log(f"MCC          = {matthews_corrcoef(test_labels, preds):.4f}")

print(classification_report(test_labels, preds, target_names=class_names))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):



PHASE 0: LOAD DATA
[12:45:52] [INFO] Client C1: 2648 samples
[12:45:52] [INFO] Client C2: 158 samples
[12:45:52] [INFO] Client C3: 1364 samples
[12:45:52] [INFO] Client C4: 765 samples
[13:06:13] [INFO] FL ROUND 1/10
[13:07:49] [INFO] FL ROUND 2/10
[13:09:24] [INFO] FL ROUND 3/10
[13:10:59] [INFO] FL ROUND 4/10
[13:12:35] [INFO] FL ROUND 5/10
[13:14:12] [INFO] FL ROUND 6/10
[13:15:49] [INFO] FL ROUND 7/10
[13:17:26] [INFO] FL ROUND 8/10
[13:19:01] [INFO] FL ROUND 9/10
[13:20:39] [INFO] FL ROUND 10/10

FINAL EVALUATION
[13:22:15] [INFO] Testing with client pipeline C1
[13:22:37] [INFO] Testing with client pipeline C2
[13:23:00] [INFO] Testing with client pipeline C3
[13:23:22] [INFO] Testing with client pipeline C4
[13:23:44] [INFO] Accuracy     = 0.9794
[13:23:44] [INFO] Precision    = 0.9795 (weighted)
[13:23:44] [INFO] Recall       = 0.9794 (weighted)
[13:23:44] [INFO] F1-score     = 0.9789
[13:23:44] [INFO] ROC-AUC      = 0.9995
[13:23:44] [INFO] PR-AUC       = 0.9724
[13:23:44] [I